# Coordinates introduced by closure

For each previous meet/join u, retain FG(u) only where u is zero.
G(h) contains G(u), but can be larger. No epsilon is used. Empty-source
closures and universal extents are separated from proper nonempty extents.
This notebook reads completed experiments, without new inference.


In [1]:
from pathlib import Path
import json
import os
import sys

root = Path(os.environ.get("LATTICE_REPOSITORY", Path.cwd())).resolve()
while not (root / "src/lattmc").exists() and root != root.parent:
    root = root.parent
if not (root / "src/lattmc").exists():
    raise RuntimeError("Set LATTICE_REPOSITORY to the checkout root.")
sys.path[:0] = [str(root), str(root / "src")]
out = Path(os.environ.get(
    "LATTICE_CLOSURE_RESULTS",
    root / "data/activation_studies/closureadded_v1"))
print("Interpreter:", sys.executable)
print("Records:", out)


Interpreter: /Users/ltsinadze/git/my_papers/.venv/bin/python
Records: /Users/ltsinadze/git/my_papers/data/activation_studies/closureadded_v1


In [2]:
from lattmc.contextstudy.closureaudit_codexgen import audit

print(audit(root, out))


{'queries': 660, 'witness_records': 44, 'replay_membership_changes': 6}


In [3]:
import pandas as pd
from IPython.display import HTML, display

summary = json.loads((out / "summary.json").read_text())
display(pd.DataFrame(summary["conditions"]).T)
columns = ["condition", "title", "original_positive", "added_positive",
           "original_count", "h_count", "additional_count"]
display(pd.DataFrame(summary["cases"])[columns])


,proper,same,expanded,became_universal,original_universal,original_empty,zero_h
sae0,36,8,28,0,53,21,0
sae8,20,12,8,3,49,41,0
sae11,25,13,12,1,49,36,0
tc0,26,15,11,0,51,33,0
tc8,30,15,15,0,48,32,0
tc11,35,18,17,0,48,27,0


,condition,title,original_positive,added_positive,original_count,h_count,additional_count
0,tc11,Research full meet,18,190,99,661,562
1,tc11,Virus/Clippers rank-1 join,2,1076,6,6,0
2,tc11,Biological rank-1 join,4,1624,3,3,0
3,tc11,Cat/dog full meet,37,255,25,25,0
4,tc11,Cat/dog rank-2/3 join,2,197,40,389,349
5,sae11,New/de rank-2/3 join,2,21,13,22,9


## Additional texts and a retained replay failure

Yellow D means collective satisfaction without a whole-query token.
Rose marks tokens meeting any requirement; bold rose marks whole-query
witnesses. R is rejection under exact replay, even if historical cached
membership was positive. The whole original decode is printed separately
from individual token fragments for readable Unicode.


In [4]:
from lattmc.contextstudy.witnessrender_codexgen import snippet

data = json.loads((out / "tc11_witnesses.json").read_text())
for row in (43, 5872, 17044, 9155):
    r = dict(next(r for r in data["records"] if r["row"] == row
                  and r["query_name"] == "h"))
    r["valid_positions"] = list(range(128))
    r["diagnostic"] = (r["partial"] or [0])[0]
    badge = r["status"]
    if badge == "D":
        badge = '<mark style="background:#FFEF9F">D</mark>'
    display(HTML(f'<h4>TC block 11, h, row {row}: {badge}</h4><p>'
                 + snippet(r, as_html=True) + '</p>'))
    print("Original decode:", r["raw_text"])
    print("Cached membership:", r["cached_member"])
    print("Exact replay membership:", r["member"])
    print("Failed requirements:", len(r["failed_coordinates"]))


Original decode: <|endoftext|> report by our group, which utilized a different goldenseal formulation [76]. Our findings also bolster recent in vitro investigations demonstrating inhibition of CYP2D6-mediated biotransformations by goldenseal extracts [59, 60]. During an evaluation of a series of single-entity herbal tea extracts, Foster et al observed that Hydrastis canadensis (goldenseal) produced the greatest percent inhibition of cDNA expressed human CYP2D6 [59]. In addition, Chatterjee and Franklin observed that goldenseal extract and its principal isoquinoline alkaloids, berberine and hydrast
Cached membership: True
Exact replay membership: True
Failed requirements: 0


Original decode: <|endoftext|> 2016, Amit Desai was promoted from senior vice president, marketing & global franchise management to exec vice president, business and marketing strategy, direct-to-consumer and global franchise management.[64]

DC Entertainment and Warner Bros. Digital Networks announced in April 2017 DC Universe digital service to be launched in 2018 with two original series.[65][66]

With frustration over DC Films not matching Marvel Studios' results and Berg wanting to step back to being a producer in January 2018, it was announced that Warner Bros. executive Walter Hamada was appointed president of DC film production.[67] After a leave of absence starting in March
Cached membership: True
Exact replay membership: True
Failed requirements: 0


Original decode: <|endoftext|> everything is on hold because there's no framework in which to judge its realism. Could Gandalf have destroyed the bridge with his staff? Who knows. A wizard did it. But it's enough to tie everything together.

On the other end of the spectrum, you have Star Trek. Very tangible world, mostly extensions of inventions we've already created, but standard human biology and physics apply. In this case, ALL science must apply and be applicable, or you have to have a VERY plausible cause for why. If Geordi wants to move a moon out of its orbit, you better explain it down to the nano-
Cached membership: True
Exact replay membership: True
Failed requirements: 0


Original decode: <|endoftext|> while Hall may find himself matched up with Welker some, a lot of the responsibility of covering him likely will fall on Kevin Barnes. The Redskins occasionally have put fellow corner Josh Wilson on the opponent’s slot receiver, however, and put Barnes on the outside. It will take a collective effort to limit Welker.

WSH tackles Sean Locklear/Willie Smith vs. NE defensive end Andre Carter

The Redskins haven’t yet announced Trent Williams’ replacement, but it will be either eight-year veteran Sean Locklear, who started two games this season at that spot with Williams injured, or
Cached membership: True
Exact replay membership: False
Failed requirements: 84


In [5]:
import numpy as np

rows = []
for r in data["records"]:
    if r["replay_changed"]:
        q = np.array(r["query"]["values"])
        maxima = np.array(r["maxima"])
        rows.append({"row": r["row"],
                     "failed": len(r["failed_coordinates"]),
                     "largest shortfall": float((q - maxima).max())})
display(pd.DataFrame(rows))


,row,failed,largest shortfall
0,5342,97,0.000030
1,9155,84,0.000025
2,9486,73,0.000018
3,10441,89,0.000013
4,14210,76,0.000017
5,25220,94,0.000015


## Interpretation limits

The research h broadens evidence discourse to organizational reporting
and explanations in fiction, with no claim that all additions are coherent.
The Cat/dog samples include both plausible entertainment/animal links and
less resolved technical or institutional passages. New/de remains lexically
dependent in its sampled additions. Equal extents on the original corpus
express finite-data implications, not causal or semantic equivalence.

Six sports h memberships fail fresh replay by small numerical shortfalls;
the original u still passes. No tolerance repairs these failures. Read the
study README and complete saved reading packets before assigning labels.
